# 08. 파이썬 기초 - pandas 심화

`04pandas기초` 가 **한 개의 표를 다루는 법** 이었다면,
이번에는 **여러 표를 합치고, 값을 가공하고, 교차표로 요약하는 법** 을 다룹니다.

넷플릭스·K-Pop·국회의원 분석 노트북에서 가장 많이 쓰이는 문법들입니다.

**다루는 내용**
1. merge — 두 표를 옆으로 합치기
2. concat — 표를 위아래로 이어붙이기
3. `.str` 접근자 — 문자열 열 한 번에 가공하기
4. explode — 한 칸에 여러 값이 든 데이터 펼치기
5. apply / map — 내가 만든 함수를 열 전체에 적용
6. pivot_table — 교차표로 요약
7. 종합 실습 — 넷플릭스 데이터

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)   # 열이 잘리지 않게
print('pandas', pd.__version__)

pandas 2.2.2


## 1. merge — 두 표를 옆으로 합치기 ⭐

**공통 열(key)을 기준으로** 두 표를 옆으로 붙입니다. 엑셀의 VLOOKUP 과 같은 역할입니다.

```
members            groups                 결과
name  group_id  +  group_id  company  =   name  group_id  company
```

`how` 옵션이 핵심입니다.

| how | 의미 |
|---|---|
| `'inner'` | 양쪽 **모두에 있는** key만 (기본값) |
| `'left'` | 왼쪽 표는 **전부 남기고**, 없으면 NaN |
| `'right'` | 오른쪽 표를 전부 남김 |
| `'outer'` | 양쪽 전부 남김 |

In [25]:
# 멤버 표
members = pd.DataFrame({
    'name':     ['지민', '수진', '태양', '하늘'],
    'group_id': ['G1', 'G1', 'G2', 'G9'],   # G9 는 groups 에 없는 값
})
print('--- members ---'); 
members


--- members ---


,name,group_id
0,지민,G1
1,수진,G1
2,태양,G2
3,하늘,G9


In [26]:

# 그룹 표
groups = pd.DataFrame({
    'group_id': ['G1', 'G2', 'G3'],
    'company':  ['하이브', 'SM', 'JYP'],
})

print('\n--- groups ---'); 
groups


--- groups ---


,group_id,company
0,G1,하이브
1,G2,SM
2,G3,JYP


In [3]:
# inner: 양쪽에 모두 있는 group_id 만 남는다 → 하늘(G9) 이 사라짐
print('=== how="inner" (기본값) ===')
print(pd.merge(members, groups, on='group_id', how='inner'))

print()
# left: 왼쪽(members) 은 전부 유지, 짝이 없으면 NaN
# 실제 분석에서는 데이터를 잃지 않으려고 left 를 훨씬 많이 쓴다
print('=== how="left" ===')
print(pd.merge(members, groups, on='group_id', how='left'))

=== how="inner" (기본값) ===
  name group_id company
0   지민       G1     하이브
1   수진       G1     하이브
2   태양       G2      SM

=== how="left" ===
  name group_id company
0   지민       G1     하이브
1   수진       G1     하이브
2   태양       G2      SM
3   하늘       G9     NaN


In [4]:
# 합친 뒤 결측 확인 = "짝을 못 찾은 행이 몇 개인가" 점검
merged = pd.merge(members, groups, on='group_id', how='left')

print('company 결측 개수:', merged['company'].isna().sum())
print('\n짝을 못 찾은 행:')
print(merged[merged['company'].isna()])

# 기준 열의 이름이 서로 다르면 left_on / right_on 사용
# pd.merge(members, groups, left_on='group_id', right_on='gid', how='left')

company 결측 개수: 1

짝을 못 찾은 행:
  name group_id company
3   하늘       G9     NaN


## 2. concat — 표를 위아래로 이어붙이기

merge 가 **옆으로**(열 추가) 라면, concat 은 **아래로**(행 추가) 입니다.

스크래핑에서 **1페이지, 2페이지 결과를 하나로 합칠 때** 쓰는 문법입니다.

In [29]:
page1 = pd.DataFrame({'title': ['A', 'B'], 'price': [1000, 2000]})
page2 = pd.DataFrame({'title': ['C', 'D'], 'price': [3000, 4000]})

# ignore_index=True : 인덱스를 0,1,2,3 으로 새로 매긴다
#                     (빼면 0,1,0,1 처럼 중복된 인덱스가 생김)
all_pages = pd.concat([page1, page2], ignore_index=True)
print(all_pages)

print('\n--- ignore_index 를 빼면 ---')
print(pd.concat([page1, page2]))

  title  price
0     A   1000
1     B   2000
2     C   3000
3     D   4000

--- ignore_index 를 빼면 ---
  title  price
0     A   1000
1     B   2000
0     C   3000
1     D   4000


In [8]:
# 실전 패턴: 반복문으로 페이지를 모아 리스트에 담고, 마지막에 한 번만 concat
frames = []
for page in range(1, 4):
    # 실제로는 여기서 requests + BeautifulSoup 로 스크래핑
    one = pd.DataFrame({'page': [page] * 2, 'item': [f'item{page}-1', f'item{page}-2']})
    frames.append(one)

result = pd.concat(frames, ignore_index=True)
print(result)
print('\n총', len(result), '행')

   page     item
0     1  item1-1
1     1  item1-2
2     2  item2-1
3     2  item2-2
4     3  item3-1
5     3  item3-2

총 6 행


## 3. `.str` 접근자 — 문자열 열을 한 번에 가공 ⭐

문자열이 든 열에 `.str` 을 붙이면 **모든 행에 문자열 메서드가 한 번에 적용** 됩니다.
반복문을 쓸 필요가 없습니다.

스크래핑한 데이터는 공백·단위·기호가 섞여 있으므로 이 정제 과정이 거의 항상 필요합니다.

In [9]:
raw = pd.DataFrame({
    'name':  ['  아이유  ', '방탄소년단', '  블랙핑크'],
    'genre': ['Ballad, Pop', 'Hip-Hop, Pop', 'Dance, Pop'],
    'price': ['25,000원', '30,000원', '28,000원'],
})

# strip()   : 앞뒤 공백 제거
raw['name'] = raw['name'].str.strip()

# replace() : 문자 치환 → 숫자로 변환
#   ',' 와 '원' 을 없앤 뒤 astype(int) 로 정수 변환
raw['price_num'] = raw['price'].str.replace(',', '').str.replace('원', '').astype(int)

print(raw)
print('\n가격 평균:', raw['price_num'].mean())

    name         genre    price  price_num
0    아이유   Ballad, Pop  25,000원      25000
1  방탄소년단  Hip-Hop, Pop  30,000원      30000
2   블랙핑크    Dance, Pop  28,000원      28000

가격 평균: 27666.666666666668


In [11]:
# contains() : 특정 문자열 포함 여부 → 필터링에 사용
print('Pop 이 들어간 행:')
print(raw[raw['genre'].str.contains('Pop')])


Pop 이 들어간 행:
    name         genre    price  price_num
0    아이유   Ballad, Pop  25,000원      25000
1  방탄소년단  Hip-Hop, Pop  30,000원      30000
2   블랙핑크    Dance, Pop  28,000원      28000


In [14]:

print()
# split()    : 쪼개기. expand=True 면 여러 열로 펼쳐진다
print('첫 번째 장르만 추출:')
print(raw['genre'])
print(raw['genre'].str.split(', '))
print(raw['genre'].str.split(', ').str[0])



첫 번째 장르만 추출:
0     Ballad, Pop
1    Hip-Hop, Pop
2      Dance, Pop
Name: genre, dtype: object
0     [Ballad, Pop]
1    [Hip-Hop, Pop]
2      [Dance, Pop]
Name: genre, dtype: object
0     Ballad
1    Hip-Hop
2      Dance
Name: genre, dtype: object


In [15]:

print()
print('여러 열로 펼치기 (expand=True):')
print(raw['genre'].str.split(', ', expand=True))


여러 열로 펼치기 (expand=True):
         0    1
0   Ballad  Pop
1  Hip-Hop  Pop
2    Dance  Pop


## 4. explode — 한 칸에 여러 값이 든 데이터 펼치기 ⭐

`"Drama, Comedy, Action"` 처럼 **한 칸에 여러 값이 쉼표로 묶여 있으면 집계를 할 수 없습니다.**

`.str.split()` 으로 리스트를 만들고 `.explode()` 로 **한 값당 한 행** 으로 펼칩니다.

```
1행: [Drama, Comedy]   →   1행: Drama
                           1행: Comedy
```

넷플릭스 장르 분석, K-Pop 그룹 분석의 핵심 문법입니다.

In [24]:
movies = pd.DataFrame({
    'title':     ['영화A', '영화B'],
    'listed_in': ['Drama, Comedy', 'Action, Drama, Thriller'],
})

# 1단계: 문자열 → 리스트
movies['genre'] = movies['listed_in'].str.split(', ')
print('--- split 후 (한 칸에 리스트가 들어있다) ---')
print(len(movies),'행')
movies


--- split 후 (한 칸에 리스트가 들어있다) ---
2 행


,title,listed_in,genre
0,영화A,"Drama, Comedy","[Drama, Comedy]"
1,영화B,"Action, Drama, Thriller","[Action, Drama, Thriller]"


In [22]:

# 2단계: 리스트 → 행으로 펼치기
exploded = movies.explode('genre')
print('\n--- explode 후 (한 장르당 한 행) ---')
print(len(movies),'행')
print(len(exploded),'행')
exploded


--- explode 후 (한 장르당 한 행) ---
2 행
5 행


,title,listed_in,genre
0,영화A,"Drama, Comedy",Drama
0,영화A,"Drama, Comedy",Comedy
1,영화B,"Action, Drama, Thriller",Action
1,영화B,"Action, Drama, Thriller",Drama
1,영화B,"Action, Drama, Thriller",Thriller


In [23]:
# 펼친 뒤에야 value_counts() 로 장르별 개수를 셀 수 있다
print('장르별 작품 수:')
print(exploded['genre'].value_counts())

# 인덱스가 0,0,1,1,1 로 중복된 점에 주의 (원래 어느 행에서 왔는지 표시)
print('\n인덱스:', list(exploded.index))
print('→ reset_index(drop=True) 로 새로 매길 수 있다')
exploded.reset_index(drop=True, inplace=True)
exploded


장르별 작품 수:
genre
Drama       2
Comedy      1
Action      1
Thriller    1
Name: count, dtype: int64

인덱스: [0, 0, 1, 1, 1]
→ reset_index(drop=True) 로 새로 매길 수 있다


,title,listed_in,genre
0,영화A,"Drama, Comedy",Drama
1,영화A,"Drama, Comedy",Comedy
2,영화B,"Action, Drama, Thriller",Action
3,영화B,"Action, Drama, Thriller",Drama
4,영화B,"Action, Drama, Thriller",Thriller


## 5. apply / map — 내가 만든 함수를 적용

`.str` 이나 사칙연산으로 해결되지 않는 **복잡한 규칙** 은 함수를 만들어 적용합니다.

- `Series.apply(함수)` : 각 **값** 에 함수 적용
- `DataFrame.apply(함수, axis=1)` : 각 **행 전체** 에 함수 적용 (여러 열을 같이 봐야 할 때)

In [30]:
books = pd.DataFrame({
    'title': ['파이썬 입문', '데이터 분석', '웹 스크래핑', '머신러닝'],
    'price': [25000, 30000, 28000, 33000],
    'pages': [300, 450, 380, 520],
})


def price_grade(price):
    """가격을 등급 문자열로 변환한다.

    Args:
        price (int): 책 가격

    Returns:
        str: '저가' / '중가' / '고가'
    """
    if price < 27000:
        return '저가'
    elif price < 31000:
        return '중가'
    return '고가'


# Series.apply : price 열의 각 '값' 에 함수를 적용
books['grade'] = books['price'].apply(price_grade)
books

,title,price,pages,grade
0,파이썬 입문,25000,300,저가
1,데이터 분석,30000,450,중가
2,웹 스크래핑,28000,380,중가
3,머신러닝,33000,520,고가


In [31]:
# DataFrame.apply(axis=1) : 각 '행' 을 통째로 받아 여러 열을 함께 사용
#   row 는 한 행을 담은 Series → row['price'], row['pages'] 로 접근
books['price_per_page'] = books.apply(
    lambda row: round(row['price'] / row['pages'], 1), axis=1
)
print(books[['title', 'price', 'pages', 'price_per_page']])

print()
# 간단한 값 치환은 map(딕셔너리) 가 더 빠르고 읽기 쉽다
grade_emoji = {'저가': '💰', '중가': '💵', '고가': '💎'}
books['icon'] = books['grade'].map(grade_emoji)
books[['title', 'grade', 'icon']]

    title  price  pages  price_per_page
0  파이썬 입문  25000    300            83.3
1  데이터 분석  30000    450            66.7
2  웹 스크래핑  28000    380            73.7
3    머신러닝  33000    520            63.5



,title,grade,icon
0,파이썬 입문,저가,💰
1,데이터 분석,중가,💵
2,웹 스크래핑,중가,💵
3,머신러닝,고가,💎


## 6. pivot_table — 교차표로 요약 ⭐

`groupby` 가 **한 방향 집계** 라면, `pivot_table` 은 **행·열 두 방향 교차표** 를 만듭니다.

```
        index=행 기준        columns=열 기준
pivot_table(index='연도', columns='종류', values='제목', aggfunc='count')
```

히트맵 시각화의 입력 데이터가 바로 이 형태입니다.

In [35]:
sales = pd.DataFrame({
    'year':    [2022, 2022, 2022, 2023, 2023, 2023, 2024],
    'type':    ['Movie', 'TV', 'TV', 'Movie', 'TV', 'Movie', 'TV'],
    'title':   ['a', 'b', 'b2', 'c', 'd', 'e', 'f'],
    'revenue': [100, 200, 150, 250, 120, 300, 180],
})
sales


,year,type,title,revenue
0,2022,Movie,a,100
1,2022,TV,b,200
2,2022,TV,b2,150
3,2023,Movie,c,250
4,2023,TV,d,120
5,2023,Movie,e,300
6,2024,TV,f,180


In [36]:

# groupby: 결과가 세로로 길게 나온다
print('=== groupby ===')
sales.groupby(['year', 'type'])['revenue'].sum()


=== groupby ===


year  type 
2022  Movie    100
      TV       350
2023  Movie    550
      TV       120
2024  TV       180
Name: revenue, dtype: int64

In [38]:

print()
# pivot_table: 같은 내용이 표(교차표) 형태로 나온다
print('=== pivot_table ===')
sales.pivot_table(index='year', columns='type', values='revenue', aggfunc='sum')


=== pivot_table ===


type,Movie,TV
year,,
2022,100.0,350.0
2023,550.0,120.0
2024,NaN,180.0


In [39]:
# aggfunc 로 집계 방법 지정: 'count', 'mean', 'sum', 'max' ...
# fill_value 로 빈 칸(NaN)을 0 으로 채운다
pt = sales.pivot_table(
    index='year',
    columns='type',
    values='title',
    aggfunc='count',
    fill_value=0,
)
print('연도별 종류별 작품 수:')
pt

# 이 표는 그대로 그래프로 그릴 수 있다
# pt.plot(kind='bar')

연도별 종류별 작품 수:


type,Movie,TV
year,,
2022,1,2
2023,2,1
2024,0,1


## 7. 종합 실습 — 넷플릭스 데이터

지금까지 배운 `.str` → `explode` → `value_counts` → `pivot_table` 을 실제 데이터에 적용합니다.

In [15]:
# 이 노트북이 python_basic/ 에 있으므로 상위 폴더의 data/ 를 가리킨다
nf = pd.read_csv('../data/netflix_titles.csv')

print('행 x 열:', nf.shape)
print('\n컬럼:', list(nf.columns))
print('\n--- 앞부분 ---')
print(nf[['type', 'title', 'release_year', 'listed_in']].head(3))

행 x 열: (8807, 12)

컬럼: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

--- 앞부분 ---
      type                 title  release_year  \
0    Movie  Dick Johnson Is Dead          2020   
1  TV Show         Blood & Water          2021   
2  TV Show             Ganglands          2021   

                                           listed_in  
0                                      Documentaries  
1    International TV Shows, TV Dramas, TV Mysteries  
2  Crime TV Shows, International TV Shows, TV Act...  


In [16]:
# listed_in 은 "Drama, Comedy" 처럼 여러 장르가 한 칸에 묶여 있다
print('원본 listed_in 예시:')
print(nf['listed_in'].head(3).to_list())

# split → explode → value_counts 로 장르별 작품 수 집계
genres = nf['listed_in'].str.split(', ').explode()

print('\n=== 장르 TOP 10 ===')
print(genres.value_counts().head(10))

원본 listed_in 예시:
['Documentaries', 'International TV Shows, TV Dramas, TV Mysteries', 'Crime TV Shows, International TV Shows, TV Action & Adventure']

=== 장르 TOP 10 ===
listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [17]:
# pivot_table 로 '최근 연도별 / 종류별 작품 수' 교차표 만들기
recent = nf[nf['release_year'] >= 2018]

pt = recent.pivot_table(
    index='release_year',
    columns='type',
    values='show_id',
    aggfunc='count',
    fill_value=0,
)
print(pt)

print('\n→ 이 표를 그대로 pt.plot(kind="bar") 하면 막대그래프가 된다')

type          Movie  TV Show
release_year                
2018            767      380
2019            633      397
2020            517      436
2021            277      315

→ 이 표를 그대로 pt.plot(kind="bar") 하면 막대그래프가 된다


In [18]:
# merge 실습: 장르별 작품 수 표를 만들고, 별도의 설명 표와 합치기
genre_count = genres.value_counts().reset_index()
genre_count.columns = ['genre', 'count']

genre_info = pd.DataFrame({
    'genre': ['Dramas', 'Comedies', 'Documentaries'],
    'korean': ['드라마', '코미디', '다큐멘터리'],
})

# how='left' → 왼쪽(genre_count) 은 전부 남고, 설명이 없으면 NaN
joined = pd.merge(genre_count.head(6), genre_info, on='genre', how='left')
print(joined)

                    genre  count korean
0    International Movies   2752    NaN
1                  Dramas   2427    드라마
2                Comedies   1674    코미디
3  International TV Shows   1351    NaN
4           Documentaries    869  다큐멘터리
5      Action & Adventure    859    NaN


## 정리

| 문법 | 하는 일 | 언제 |
|---|---|---|
| `pd.merge(A, B, on=, how=)` | 두 표를 **옆으로** 합침 | 다른 표의 정보를 가져올 때 |
| `pd.concat([A, B], ignore_index=True)` | 표를 **아래로** 이어붙임 | 페이지별 스크래핑 결과 합칠 때 |
| `.str.strip() / replace() / contains() / split()` | 문자열 열 일괄 가공 | 스크래핑 데이터 정제 |
| `.explode()` | 한 칸의 여러 값을 행으로 펼침 | 장르·출연진 집계 |
| `.apply(함수)` | 사용자 함수 적용 | 복잡한 규칙의 파생 열 |
| `.pivot_table(index=, columns=)` | 교차표 요약 | 히트맵·비교표 |

**꼭 기억할 것**
- `merge` 는 `how='left'` 를 기본으로 생각하기 (데이터를 잃지 않음)
- 합친 뒤에는 항상 `isna().sum()` 으로 짝을 못 찾은 행 확인
- 쉼표로 묶인 열은 `.str.split()` → `.explode()` 로 펼쳐야 집계 가능

다음: `09python_basic_numpy.ipynb` (pandas 밑에 깔린 계산 엔진)